# SHS generation + analysis

In [ ]:
# Sequence & structure 
RNA_SEQ =   "AUUAGCCGGUUGAAAGACGAGGCACCCUUCUU"
STRUCTURE = "()()()()................()()()()"  # overrides the predictor if set
STRUCTURE_PREDICTOR = "rnafold"          # rnafold, spotrna, rnaformer

# MSA size / reproducibility 
N = 10000
SEED = 42

# Mutation parameters
MUTATION_RATE_UNPAIRED = 0.2
MUTATION_RATE_PAIRED   = 0.2
PAIR_MUTATION_APPROACH = "covariance" # covariance | watson_crick | original

# Base-triple / multiplet co-mutation (0.0 disables)
TRIPLET_PROB           = 0.0 # 0.0
TRIPLET_KEEP_PROB      = 0.99 # 0.99
WOBBLE_PROB            = 0.0 # 0.1 

# insertions / deletions - general
MAX_INSERTION_FRACTION = 0.0 # 0.1
MAX_DELETION_FRACTION  = 0.0 # 0.1
# paired
STEM_SINGLE_INSERTION_PROB  = 0.0
STEM_LONG_INSERTION_PROB    = 0.00 # 0.05
STEM_PAIR_DELETION_PROB     = 0.0
# unpaired
LOOP_SINGLE_INSERTION_PROB = 0.0 # 0.2
LOOP_SINGLE_DELETION_PROB  = 0.0 # 0.8
LOOP_LONG_INSERTION_PROB   = 0.00 # 0.05
LOOP_LONG_DELETION_PROB    = 0.05 # 0.05

# Output
OUTPUT_JSON_DIR = "custom_msa_json_output"
PDB_ID = "None"

assert(MUTATION_RATE_UNPAIRED + LOOP_SINGLE_DELETION_PROB <= 1.0)
assert(STEM_SINGLE_INSERTION_PROB + STEM_LONG_INSERTION_PROB <= 1.0)

assert(MUTATION_RATE_PAIRED + STEM_PAIR_DELETION_PROB <= 1.0)
assert(STEM_SINGLE_INSERTION_PROB + STEM_LONG_INSERTION_PROB <= 1.0)

## Setup

In [ ]:
import importlib
import os
import sys
from argparse import Namespace
from pathlib import Path

%matplotlib inline

_candidates = [Path.cwd(), Path.cwd() / "SHS-Generator"]
SHS_DIR = next((p for p in _candidates if (p / "shs_generator.py").exists()), None)
if SHS_DIR is None:
    raise FileNotFoundError("Could not find shs_generator.py.")
SHS_DIR = SHS_DIR.resolve()
os.chdir(SHS_DIR)
if str(SHS_DIR) not in sys.path:
    sys.path.insert(0, str(SHS_DIR))

import shs_generator
import shs_analyzer
importlib.reload(shs_generator)
importlib.reload(shs_analyzer)

print("SHS-Generator dir:", SHS_DIR)

## Generate the SHS

In [ ]:

STRUCTURE = "".join(["\"" + c + "\"" if c.isdigit() else c for c in STRUCTURE])

args = Namespace(
    structure_predictor        = None if STRUCTURE else STRUCTURE_PREDICTOR,
    structure                  = STRUCTURE,
    rna_seq                    = RNA_SEQ,
    protein_seq                = "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQ",  # placeholder; AF3 JSON needs a protein chain
    input_json_path            = None,
    pdb_id                     = PDB_ID,
    output_json_dir            = OUTPUT_JSON_DIR,
    N                          = N,
    mutation_rate_unpaired     = MUTATION_RATE_UNPAIRED,
    mutation_rate_paired       = MUTATION_RATE_PAIRED,
    pair_mutation_approach     = PAIR_MUTATION_APPROACH,
    stem_single_insertion_prob = STEM_SINGLE_INSERTION_PROB,
    stem_long_insertion_prob   = STEM_LONG_INSERTION_PROB,
    stem_pair_deletion_prob    = STEM_PAIR_DELETION_PROB,
    loop_single_insertion_prob = LOOP_SINGLE_INSERTION_PROB,
    loop_single_deletion_prob  = LOOP_SINGLE_DELETION_PROB,
    loop_long_insertion_prob   = LOOP_LONG_INSERTION_PROB,
    loop_long_deletion_prob    = LOOP_LONG_DELETION_PROB,
    max_insertion_fraction     = MAX_INSERTION_FRACTION,
    max_deletion_fraction      = MAX_DELETION_FRACTION,
    triplet_prob               = TRIPLET_PROB,
    triplet_keep_prob          = TRIPLET_KEEP_PROB,
    wobble_prob                = WOBBLE_PROB,
    seed                       = SEED,
    max_chains                 = None,
    plot                       = False,
    print_msa                  = False,
    show_plot                  = False,
)

generator = shs_generator.MsaGenerator(args)
# write=False: build the AF3 JSON and return it for in-memory analysis without
# touching the filesystem. Pass write=True (the default) to also save it.
generated_json = generator.process(write=False)
print("Generated AF3 JSON with", len(generated_json["sequences"]), "chains; name:", generated_json["name"])

## Analyze & render

In [ ]:
import json

data = shs_analyzer.MsaData.from_text(json.dumps(generated_json), pairs=generator.pairs)
feat = shs_analyzer.Features(data)

print(f"Query length: {len(data.seq)}  "
   f"|  MSA rows: {data.aligned.shape[0] + 1}  "
   f"|  covariance shape: {feat.covariance.shape}")

shs_analyzer.plot_covariance(data.seq, feat.covariance, title="Recovered (covariation)", show_values=True)
shs_analyzer.plot_covariance_classification(data.seq, feat.covariance, pairs=data.pairs)
shs_analyzer.plot_deletion_rate(data.seq, feat.col_deletion_rate, pairs=generator.pairs)

print(data.seq)
for row in data.aligned[:10]:
    print("".join(row))

## Recovered parameters

In [ ]:
import pandas as pd

# Show full cell contents so the `detail` strings aren't truncated.
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# Known generator input per recovered parameter name (the ground truth each
# estimator tries to reproduce). Extend this as you add estimators.
KNOWN_INPUTS = {
    "n_sequences":            N,
    "mutation_rate_unpaired": MUTATION_RATE_UNPAIRED,
    "mutation_rate_paired":   MUTATION_RATE_PAIRED,
}

report = shs_analyzer.analyze_data(data)


def _fmt_detail(detail):
    # Turn {'unpaired_cols': 14, ...} into a readable "unpaired_cols=14 · ..." line.
    return " · ".join(f"{k}={v}" for k, v in detail.items())


rows = []
for name, est in report.items():
    known = KNOWN_INPUTS.get(name)
    # est.value != est.value is True only for NaN -> no error there.
    error = abs(est.value - known) if (known is not None and est.value == est.value) else None
    rows.append({
        "parameter": name,
        "known":     known,
        "recovered": round(est.value, 4),
        "abs_error": round(error, 4) if error is not None else None,
        "detail":    _fmt_detail(est.detail),
    })

pd.DataFrame(rows).set_index("parameter")